
# Sección 3: Colección y Descripción de Datos

## Proyecto: Relación entre resultados ICFES y acceso a internet por municipio en Colombia

**Curso:** Procesamiento de Datos a Gran Escala  
**Universidad:** Pontificia Universidad Javeriana  
**Metodología:** CRISP-DM  

---

En esta sección se realiza la carga de los datos en el ambiente de trabajo Apache Spark 
y se describen los conjuntos de datos según sus tipos, atributos y contenido general.
Los datos provienen de la API pública de datos.gov.co (Socrata) y corresponden a tres 
fuentes oficiales del gobierno colombiano.

In [1]:
import os
import sys

os.environ["SPARK_HOME"] = "/home/estudiante/spark"
os.environ["PYSPARK_PYTHON"] = sys.executable

sys.path.insert(0, "/home/estudiante/spark/python")
sys.path.insert(0, "/home/estudiante/spark/python/lib/py4j-0.10.9.7-src.zip")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Seccion3_Coleccion_Descripcion") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Sesión Spark iniciada correctamente")
print(f"Versión de Spark: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 18:58:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sesión Spark iniciada correctamente
Versión de Spark: 3.5.2


## 3.1 Carga de los datos

Los datos fueron descargados desde la API Socrata de datos.gov.co en formato JSONL 
y se cargan en Apache Spark para su procesamiento distribuido. Se utilizan tres 
conjuntos de datos oficiales:

| Dataset | Fuente | ID Socrata | Descripción |
|---|---|---|---|
| ICFES Saber 11 | ICFES | kgxf-xxbe | Resultados individuales prueba Saber 11 |
| Internet por municipio | MinTIC | n48w-gutb | Accesos a internet fijo por municipio |
| Cobertura educativa | MEN | nudc-7mev | Indicadores educativos por municipio |

In [2]:
RAW_DIR = "data/raw_json"

df_icfes       = spark.read.json(f"{RAW_DIR}/icfes")
df_internet    = spark.read.json(f"{RAW_DIR}/internet")
df_bachillerato = spark.read.json(f"{RAW_DIR}/bachillerato")

print("=" * 55)
print("RESUMEN DE CARGA DE DATOS")
print("=" * 55)
print(f"{'Dataset':<25} {'Registros':>10} {'Columnas':>10}")
print("-" * 55)
print(f"{'ICFES Saber 11':<25} {df_icfes.count():>10,} {len(df_icfes.columns):>10}")
print(f"{'Internet por municipio':<25} {df_internet.count():>10,} {len(df_internet.columns):>10}")
print(f"{'Cobertura educativa':<25} {df_bachillerato.count():>10,} {len(df_bachillerato.columns):>10}")
print("=" * 55)
print("\nNota: datos cargados en modo muestra (5.000 registros por dataset)")

RESUMEN DE CARGA DE DATOS
Dataset                    Registros   Columnas
-------------------------------------------------------
ICFES Saber 11                 5,000         51
Internet por municipio         5,000         12
Cobertura educativa            5,000         41

Nota: datos cargados en modo muestra (5.000 registros por dataset)


## 3.2 Tipos de datos

A continuación se presenta el esquema inferido por Spark para cada dataset,
mostrando el nombre y tipo de cada columna.

In [3]:
print("=" * 55)
print("ESQUEMA — Dataset ICFES Saber 11")
print("=" * 55)
df_icfes.printSchema()

ESQUEMA — Dataset ICFES Saber 11
root
 |-- cole_area_ubicacion: string (nullable = true)
 |-- cole_bilingue: string (nullable = true)
 |-- cole_calendario: string (nullable = true)
 |-- cole_caracter: string (nullable = true)
 |-- cole_cod_dane_establecimiento: string (nullable = true)
 |-- cole_cod_dane_sede: string (nullable = true)
 |-- cole_cod_depto_ubicacion: string (nullable = true)
 |-- cole_cod_mcpio_ubicacion: string (nullable = true)
 |-- cole_codigo_icfes: string (nullable = true)
 |-- cole_depto_ubicacion: string (nullable = true)
 |-- cole_genero: string (nullable = true)
 |-- cole_jornada: string (nullable = true)
 |-- cole_mcpio_ubicacion: string (nullable = true)
 |-- cole_naturaleza: string (nullable = true)
 |-- cole_nombre_establecimiento: string (nullable = true)
 |-- cole_nombre_sede: string (nullable = true)
 |-- cole_sede_principal: string (nullable = true)
 |-- desemp_ingles: string (nullable = true)
 |-- estu_cod_depto_presentacion: string (nullable = true)
 |

In [8]:
print("=" * 60)
print("ESQUEMA - Dataset Internet por municipio")
print("=" * 60)
df_internet.printSchema()

ESQUEMA - Dataset Internet por municipio
root
 |-- _2017: string (nullable = true)
 |-- _2018: string (nullable = true)
 |-- _2019: string (nullable = true)
 |-- a_o_2014: string (nullable = true)
 |-- a_o_2015: string (nullable = true)
 |-- a_o_2016: string (nullable = true)
 |-- a_o_2020: string (nullable = true)
 |-- a_o_2021: string (nullable = true)
 |-- a_o_2022: string (nullable = true)
 |-- a_o_2023: string (nullable = true)
 |-- a_o_2024: string (nullable = true)
 |-- institucion_educativa: string (nullable = true)
 |-- municipio: string (nullable = true)



In [4]:
print("=" * 55)
print("ESQUEMA — Dataset Internet por municipio")
print("=" * 55)
df_internet.printSchema()

ESQUEMA — Dataset Internet por municipio
root
 |-- anno: string (nullable = true)
 |-- cod_departamento: string (nullable = true)
 |-- cod_municipio: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- no_de_accesos: string (nullable = true)
 |-- proveedor: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- tecnologia: string (nullable = true)
 |-- trimestre: string (nullable = true)
 |-- velocidad_bajada: string (nullable = true)
 |-- velocidad_subida: string (nullable = true)



In [5]:
print("=" * 55)
print("ESQUEMA — Dataset Cobertura educativa")
print("=" * 55)
df_bachillerato.printSchema()

ESQUEMA — Dataset Cobertura educativa
root
 |-- a_o: string (nullable = true)
 |-- aprobaci_n: string (nullable = true)
 |-- aprobaci_n_media: string (nullable = true)
 |-- aprobaci_n_primaria: string (nullable = true)
 |-- aprobaci_n_secundaria: string (nullable = true)
 |-- aprobaci_n_transici_n: string (nullable = true)
 |-- c_digo_departamento: string (nullable = true)
 |-- c_digo_etc: string (nullable = true)
 |-- c_digo_municipio: string (nullable = true)
 |-- cobertura_bruta: string (nullable = true)
 |-- cobertura_bruta_media: string (nullable = true)
 |-- cobertura_bruta_primaria: string (nullable = true)
 |-- cobertura_bruta_secundaria: string (nullable = true)
 |-- cobertura_bruta_transici_n: string (nullable = true)
 |-- cobertura_neta: string (nullable = true)
 |-- cobertura_neta_media: string (nullable = true)
 |-- cobertura_neta_primaria: string (nullable = true)
 |-- cobertura_neta_secundaria: string (nullable = true)
 |-- cobertura_neta_transici_n: string (nullable = t

## 3.3 Comprensión del significado de cada atributo

### Dataset 1: ICFES Saber 11
Contiene los resultados individuales de los estudiantes colombianos en la prueba 
Saber 11. Cada fila representa un estudiante en una aplicación del examen.

| Atributo | Tipo | Descripción |
|---|---|---|
| `periodo` | String | Año y semestre del examen (ej. 20231 = primer semestre 2023) |
| `estu_consecutivo` | String | Identificador único del estudiante |
| `estu_genero` | String | Género del estudiante (M/F) |
| `estu_fechanacimiento` | String | Fecha de nacimiento del estudiante |
| `cole_mcpio_ubicacion` | String | Municipio donde está ubicado el colegio |
| `cole_depto_ubicacion` | String | Departamento donde está ubicado el colegio |
| `cole_naturaleza` | String | Naturaleza del colegio (oficial / no oficial) |
| `cole_calendario` | String | Calendario del colegio (A / B / otro) |
| `cole_bilingue` | String | Indica si el colegio es bilingüe (S/N) |
| `cole_caracter` | String | Carácter académico del colegio |
| `cole_jornada` | String | Jornada escolar (mañana, tarde, noche, completa) |
| `cole_area_ubicacion` | String | Zona del colegio (urbano / rural) |
| `fami_estratovivienda` | String | Estrato socioeconómico de la familia |
| `fami_educacionmadre` | String | Nivel educativo de la madre |
| `fami_educacionpadre` | String | Nivel educativo del padre |
| `fami_tieneinternet` | String | Si la familia tiene internet en casa (Si/No) |
| `fami_tienecomputador` | String | Si la familia tiene computador (Si/No) |
| `fami_tieneautomovil` | String | Si la familia tiene automóvil (Si/No) |
| `fami_tienelavadora` | String | Si la familia tiene lavadora (Si/No) |
| `fami_personashogar` | String | Número de personas en el hogar |
| `fami_cuartoshogar` | String | Número de cuartos en el hogar |
| `desemp_ingles` | String | Nivel de desempeño en inglés (A1, A2, B1, B+) |
| `punt_ingles` | String | Puntaje en inglés |
| `punt_matematicas` | String | Puntaje en matemáticas |
| `estu_privado_libertad` | String | Si el estudiante está privado de libertad (S/N) |

### Dataset 2: Internet por municipio
Contiene información sobre accesos a internet fijo por municipio, proveedor 
y segmento de usuario. Cada fila representa un registro de accesos en un 
trimestre específico.

| Atributo | Tipo | Descripción |
|---|---|---|
| `anno` | String | Año del registro |
| `trimestre` | String | Trimestre del año (1, 2, 3, 4) |
| `municipio` | String | Nombre del municipio |
| `cod_municipio` | String | Código DANE del municipio |
| `departamento` | String | Nombre del departamento |
| `cod_departamento` | String | Código DANE del departamento |
| `proveedor` | String | Empresa proveedora del servicio de internet |
| `tecnologia` | String | Tecnología usada (fibra, cable, inalámbrica, etc.) |
| `segmento` | String | Segmento de usuario (residencial estrato 1-6, corporativo) |
| `velocidad_bajada` | String | Velocidad de descarga en Mbps |
| `velocidad_subida` | String | Velocidad de subida en Mbps |
| `no_de_accesos` | String | Número de accesos en ese municipio/proveedor/segmento |

### Dataset 3: Cobertura educativa por municipio
Contiene indicadores educativos agregados por municipio y año, publicados 
por el Ministerio de Educación Nacional.

| Atributo | Tipo | Descripción |
|---|---|---|
| `a_o` | String | Año del registro |
| `municipio` | String | Nombre del municipio |
| `c_digo_municipio` | String | Código DANE del municipio |
| `departamento` | String | Nombre del departamento |
| `cobertura_neta` | String | Tasa de cobertura neta total (%) |
| `cobertura_neta_primaria` | String | Cobertura neta en primaria (%) |
| `cobertura_neta_secundaria` | String | Cobertura neta en secundaria (%) |
| `cobertura_neta_media` | String | Cobertura neta en media (%) |
| `cobertura_bruta` | String | Tasa de cobertura bruta total (%) |
| `deserci_n` | String | Tasa de deserción escolar total (%) |
| `deserci_n_primaria` | String | Deserción en primaria (%) |
| `deserci_n_secundaria` | String | Deserción en secundaria (%) |
| `aprobaci_n` | String | Tasa de aprobación escolar total (%) |
| `reprobaci_n` | String | Tasa de reprobación escolar total (%) |
| `repitencia` | String | Tasa de repitencia escolar total (%) |
| `sedes_conectadas_a_internet` | String | Porcentaje de sedes educativas con internet (%) |
| `tasa_matriculaci_n_5_16` | String | Tasa de matriculación población 5-16 años (%) |
| `poblaci_n_5_16` | String | Población en edad escolar (5-16 años) |
| `tama_o_promedio_de_grupo` | String | Tamaño promedio de grupo escolar |

## 3.4 Descripción general del contenido

### 3.4.1 Dataset ICFES Saber 11

In [7]:
print("=" * 55)
print("DESCRIPCIÓN — ICFES Saber 11")
print("=" * 55)
print(f"Total registros:  {df_icfes.count():,}")
print(f"Total columnas:   {len(df_icfes.columns)}")

print("\nPeriodos disponibles:")
df_icfes.groupBy("periodo") \
    .count() \
    .orderBy("periodo") \
    .show(10)

print("Departamentos presentes:")
df_icfes.groupBy("cole_depto_ubicacion") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

DESCRIPCIÓN — ICFES Saber 11
Total registros:  5,000
Total columnas:   51

Periodos disponibles:
+-------+-----+
|periodo|count|
+-------+-----+
|  20101|   27|
|  20102|  549|
|  20111|   25|
|  20112|  541|
|  20121|   43|
|  20122|  524|
|  20131|   57|
|  20132|  448|
|  20141|   18|
|  20142|  424|
+-------+-----+
only showing top 10 rows

Departamentos presentes:
+--------------------+-----+
|cole_depto_ubicacion|count|
+--------------------+-----+
|           ANTIOQUIA|  759|
|              BOGOTA|  565|
|               VALLE|  442|
|        CUNDINAMARCA|  357|
|           ATLANTICO|  265|
|           SANTANDER|  249|
|              BOGOTÁ|  231|
|             BOLIVAR|  207|
|             CORDOBA|  167|
|              TOLIMA|  161|
+--------------------+-----+
only showing top 10 rows



### 3.4.2 Dataset Internet por municipio

In [9]:
print("=" * 55)
print("DESCRIPCIÓN — Internet por municipio")
print("=" * 55)
print(f"Total registros:  {df_internet.count():,}")
print(f"Total columnas:   {len(df_internet.columns)}")

print("\nAños disponibles:")
df_internet.groupBy("anno") \
    .count() \
    .orderBy("anno") \
    .show()

print("Top 10 municipios con más registros:")
df_internet.groupBy("municipio") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

print("Tecnologías de acceso:")
df_internet.groupBy("tecnologia") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

DESCRIPCIÓN — Internet por municipio
Total registros:  5,000
Total columnas:   12

Años disponibles:
+----+-----+
|anno|count|
+----+-----+
|2021| 1129|
|2022| 2155|
|2023| 1716|
+----+-----+

Top 10 municipios con más registros:
+------------------+-----+
|         municipio|count|
+------------------+-----+
|      BOGOTÁ, D.C.|  146|
|              CALI|  113|
|          MEDELLÍN|   99|
|     VILLAVICENCIO|   77|
|       BUCARAMANGA|   77|
|      BARRANQUILLA|   75|
|         CARTAGENA|   73|
|SAN JOSÉ DE CÚCUTA|   63|
|           PEREIRA|   62|
|            IBAGUÉ|   61|
+------------------+-----+
only showing top 10 rows

Tecnologías de acceso:
+--------------------+-----+
|          tecnologia|count|
+--------------------+-----+
|               CABLE| 1446|
|FIBER TO THE HOME...| 1269|
|                XDSL|  744|
|HYBRID FIBER COAX...|  460|
|OTRAS TECNOLOGÍAS...|  380|
|           SATELITAL|  252|
|                WIFI|  200|
|OTRAS TECNOLOGÍAS...|  159|
|FIBER TO THE BUIL...|  

### 3.4.3 Dataset Cobertura educativa

In [10]:
print("=" * 55)
print("DESCRIPCIÓN — Cobertura educativa")
print("=" * 55)
print(f"Total registros:  {df_bachillerato.count():,}")
print(f"Total columnas:   {len(df_bachillerato.columns)}")

print("\nAños disponibles:")
df_bachillerato.groupBy("a_o") \
    .count() \
    .orderBy("a_o") \
    .show()

print("Top 10 departamentos con más registros:")
df_bachillerato.groupBy("departamento") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

DESCRIPCIÓN — Cobertura educativa
Total registros:  5,000
Total columnas:   41

Años disponibles:
+----+-----+
| a_o|count|
+----+-----+
|2011| 1122|
|2012| 1122|
|2013| 1122|
|2014| 1122|
|2015|  504|
|2019|    1|
|2020|    2|
|2021|    1|
|2022|    4|
+----+-----+

Top 10 departamentos con más registros:
+------------------+-----+
|      departamento|count|
+------------------+-----+
|         Antioquia|  625|
|            Boyacá|  617|
|      Cundinamarca|  509|
|         Santander|  349|
|            Nariño|  258|
|           Bolívar|  230|
|             Cauca|  210|
|            Tolima|  188|
|   Valle del Cauca|  168|
|Norte de Santander|  160|
+------------------+-----+
only showing top 10 rows



## 3.5 Estadísticos descriptivos

En esta sección se presentan los estadísticos básicos de las variables 
numéricas más relevantes de cada dataset.

In [12]:
print("Estadísticos descriptivos — ICFES Saber 11")
print("(variables de puntaje)")

df_icfes.select(
    F.col("punt_matematicas").cast("double"),
    F.col("punt_ingles").cast("double")
).describe().show()

Estadísticos descriptivos — ICFES Saber 11
(variables de puntaje)
+-------+------------------+------------------+
|summary|  punt_matematicas|       punt_ingles|
+-------+------------------+------------------+
|  count|              5000|              5000|
|   mean|48.884170000000005| 48.12052599999996|
| stddev|11.479404033942487|12.136748442473179|
|    min|               0.0|              -1.0|
|    max|             100.0|            102.61|
+-------+------------------+------------------+



In [18]:
print("Estadísticos descriptivos — Internet por municipio")
#Se remplaza la , por punto para que pueda leerlo como numero, mas adelante en el punto 7 se hace la respectiva transformacion.

df_internet.select(
    F.regexp_replace(F.col("velocidad_bajada"), ",", ".").cast("double").alias("velocidad_bajada"),
    F.regexp_replace(F.col("velocidad_subida"), ",", ".").cast("double").alias("velocidad_subida"),
    F.col("no_de_accesos").cast("double")
).describe().show()

Estadísticos descriptivos — Internet por municipio
+-------+-----------------+------------------+------------------+
|summary| velocidad_bajada|  velocidad_subida|     no_de_accesos|
+-------+-----------------+------------------+------------------+
|  count|             5000|              5000|              5000|
|   mean|       272.900584|219.66883399999998|           45.8632|
| stddev|4894.559562928663| 4873.452515812376|302.32801942927205|
|    min|              0.0|               0.0|               0.0|
|    max|         220300.0|          220300.0|            9566.0|
+-------+-----------------+------------------+------------------+



In [15]:
print("Estadísticos descriptivos — Cobertura educativa")

df_bachillerato.select(
    F.col("cobertura_neta").cast("double"),
    F.col("deserci_n").cast("double"),
    F.col("aprobaci_n").cast("double"),
    F.col("sedes_conectadas_a_internet").cast("double")
).describe().show()

Estadísticos descriptivos — Cobertura educativa
+-------+-----------------+------------------+-----------------+---------------------------+
|summary|   cobertura_neta|         deserci_n|       aprobaci_n|sedes_conectadas_a_internet|
+-------+-----------------+------------------+-----------------+---------------------------+
|  count|             4912|              4882|             4986|                       4975|
|   mean|85.86475614820847| 3.612931646866038|93.15219647011634|          34.95700904522615|
| stddev|15.49033122126782|2.1981227920674637|5.235874314502914|          27.30462565217377|
|    min|            0.919|            0.0252|           0.8835|                        0.0|
|    max|            139.7|             12.37|            100.0|                      100.0|
+-------+-----------------+------------------+-----------------+---------------------------+



## 3.6 Observaciones importantes

- **ICFES:** Todos los atributos son de tipo `string`. Las columnas de puntaje 
  (`punt_matematicas`, `punt_ingles`) requieren conversión a tipo numérico para análisis.
  
- **Internet:** La columna `no_de_accesos` está en formato string con posibles 
  caracteres especiales. Las velocidades usan coma como separador decimal, lo que 
  requiere limpieza antes del análisis numérico.

- **Cobertura educativa:** Los nombres de columnas tienen caracteres especiales 
  (`_o`, `_digo`, `_n`) por codificación de tildes. Esto se normalizará en la 
  fase de transformación.

- **Clave de cruce:** Los tres datasets podemos cruzarlos por `municipio` y año, 
  lo que nos podra permitir construir un análisis territorial integrado de conectividad 
  y resultados educativos.

In [19]:
spark.stop()
print("Sesión Spark cerrada correctamente.")

Sesión Spark cerrada correctamente.
